[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/05_Advanced_Topics/06_multimodal_reasoning/06_multimodal_reasoning.ipynb)

# 06. Multimodal Reasoning

**This notebook covers:**
- Chain-of-thought prompting example
- Visual reasoning pipeline
- VisProg-style program generation

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/05_Advanced_Topics/06_multimodal_reasoning"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. Chain-of-Thought Prompting

Multimodal CoT asks the model to reason step-by-step before answering.


In [ ]:
COT_TEMPLATE = """Question: {question}
Let's think step by step:
1) Identify relevant objects in the image
2) Extract attributes (color, count, spatial relations)
3) Apply logic to answer

Answer:"""

question = "How many red objects are to the left of the blue square?"
prompt = COT_TEMPLATE.format(question=question)
print(prompt)

# Simulated CoT trace (rule-based demo)
steps = [
    "Detected: 2 red circles, 1 blue square at x=0.7",
    "Red objects at x=0.2 and x=0.5 are left of blue square",
    "Count = 2",
]
for i, s in enumerate(steps, 1):
    print(f"  Step {i}: {s}")
print("Final answer: 2")

## 2. Visual Reasoning Pipeline


In [ ]:
class VisualReasoningPipeline:
    def __init__(self):
        self.objects = []

    def detect(self, image_tensor):
        # Synthetic detections on 8x8 grid
        self.objects = [
            {"label": "red_circle", "x": 0.2, "y": 0.5, "color": "red"},
            {"label": "red_circle", "x": 0.5, "y": 0.4, "color": "red"},
            {"label": "blue_square", "x": 0.75, "y": 0.5, "color": "blue"},
        ]
        return self.objects

    def answer_spatial(self, question):
        if "left of" in question and "blue" in question:
            ref = [o for o in self.objects if o["color"] == "blue"][0]
            left = [o for o in self.objects if o["x"] < ref["x"] and "red" in o["label"]]
            return len(left)
        return 0

pipe = VisualReasoningPipeline()
img = torch.rand(3, 64, 64)
dets = pipe.detect(img)
ans = pipe.answer_spatial(question)
print("Detections:", dets)
print("Answer:", ans)

## 3. VisProg-Style Program Generation


In [ ]:
PROGRAM_LIB = {
    "FIND": "find(object_type)",
    "COUNT": "count(objects)",
    "FILTER_COLOR": "filter(objects, color)",
    "LEFT_OF": "left_of(objects, reference)",
}

def generate_visprog(question):
    q = question.lower()
    program = []
    if "red" in q:
        program.append("objs = FIND('circle')")
        program.append("red = FILTER_COLOR(objs, 'red')")
    if "blue" in q:
        program.append("ref = FIND('square')")
        program.append("blue = FILTER_COLOR(ref, 'blue')")
    if "left" in q:
        program.append("result = LEFT_OF(red, blue[0])")
        program.append("answer = COUNT(result)")
    else:
        program.append("answer = COUNT(red)")
    return program

program = generate_visprog(question)
print("Generated VisProg:")
for line in program:
    print(" ", line)

# Execute program symbolically
env = {"red": [0, 1], "blue": [2], "result": [0, 1]}
exec_lines = {"answer = COUNT(result)": len(env["result"])}
print("\nExecution result:", exec_lines["answer = COUNT(result)"])

## 4. Visualize Reasoning on Synthetic Scene


In [ ]:
canvas = np.ones((8, 8, 3))
canvas[3:5, 1:3] = [1, 0, 0]
canvas[3:5, 4:6] = [1, 0, 0]
canvas[3:5, 6:8] = [0, 0, 1]

plt.imshow(canvas)
for o in dets:
    plt.scatter(o['x']*8, o['y']*8, s=120, facecolors='none', edgecolors='yellow', linewidths=2)
plt.title(f'Visual reasoning demo — answer={ans}')
plt.axis('off'); plt.show()

## Summary

Demonstrated multimodal CoT prompts, a visual reasoning pipeline, and VisProg-style programs.

**Next:** [07_text_to_image_video](../07_text_to_image_video/07_text_to_image_video.ipynb)
